                # HilbertSpace And Lookup

                This notebook documents how `HilbertSpace` combines independently defined subsystems, how interactions are added, and how bare/dressed lookup tables are built and queried.
                

                **Audience:** readers ready to move from single systems to coupled systems.

                **Prerequisites:** the shared API from notebook 01 and the meaning of dressed versus bare states.

                **Learning goals:** assemble a composite Hamiltonian, add interactions and direct operator terms, diagonalize the coupled system, build a `SpectrumLookup`, and move operators into the dressed basis.
                

                ## Outline

                1. Build a coupled pair of Kerr oscillators.
                2. Add interaction terms and an extra operator directly to the full Hilbert space.
                3. Diagonalize the system and generate bare/dressed lookup data.
                4. Transform an operator into the dressed basis.
                

In [ ]:
                using ScQubitsMimic
                using LinearAlgebra
                

In [ ]:
                kosc1 = KerrOscillator(E_osc=4.5, K=0.05, truncated_dim=6)
                kosc2 = KerrOscillator(E_osc=6.0, K=0.03, truncated_dim=6)
                hs = HilbertSpace([kosc1, kosc2])
                dims = [hilbertdim(s) for s in hs.subsystems]

                bare_summary = (
                    subsystem_dims = dims,
                    bare_dimension = hilbertdim(hs),
                    bare_evals = round.(eigenvals(hs; evals_count=6), digits=6),
                )
                bare_summary
                

In [ ]:
                g = 0.1
                add_interaction!(hs, g, [kosc1, kosc2], [s -> creation_operator(s), s -> annihilation_operator(s)])
                add_interaction!(hs, g, [kosc1, kosc2], [s -> annihilation_operator(s), s -> creation_operator(s)])

                detuning_term = 0.02 * identity_wrap(number_operator(kosc2), 2, dims)
                add_operator!(hs, detuning_term)

                (
                    interaction_count = length(hs.interactions),
                    extra_operator_count = length(hs.extra_H_terms),
                    dressed_evals = round.(diag!(hs; evals_count=8)[1], digits=6),
                )
                

In [ ]:
                lookup = generate_lookup!(hs; evals_count=12)
                (
                    lookup_built = lookup_exists(hs),
                    overlap_threshold = OVERLAP_THRESHOLD,
                    dressed_1_to_bare = bare_index(hs, 1),
                    bare_2_1_to_dressed = dressed_index(hs, 2, 1),
                    energy_dressed_2 = round(energy_by_dressed_index(hs, 2), digits=6),
                    energy_bare_1_2 = round(energy_by_bare_index(hs, 1, 2), digits=6),
                )
                

In [ ]:
                n1_full = identity_wrap(number_operator(kosc1), 1, dims)
                n1_dressed = op_in_dressed_eigenbasis(hs, n1_full; truncated_dim=5)
                round.(n1_dressed, digits=4)
                

                ## Exercise

                Replace the two Kerr oscillators with a `Transmon` and an `Oscillator`. Rebuild the lookup and identify which dressed state most closely matches bare state `(2, 1)`.
                

In [ ]:
                tmon_exercise = Transmon(EJ=30.0, EC=1.2, ng=0.0, ncut=12, truncated_dim=4)
                osc_exercise = Oscillator(E_osc=7.0, truncated_dim=6)
                hs_exercise = HilbertSpace([tmon_exercise, osc_exercise])
                add_interaction!(hs_exercise, 0.1, [tmon_exercise, osc_exercise],
                    [s -> n_operator(s), s -> annihilation_operator(s) + creation_operator(s)])
                generate_lookup!(hs_exercise; evals_count=12)
                dressed_index(hs_exercise, 2, 1)
                

                ## Pitfalls And Extensions

                `generate_lookup!` needs enough dressed states to label the bare states you care about. If a lookup query fails, raise `evals_count` and rebuild the lookup.

                `identity_wrap` is useful whenever you want to add a hand-built operator to the composite system. The helper follows the subsystem order stored on the `HilbertSpace`.
                

                ## API Covered

                `HilbertSpace`, `InteractionTerm`, `SpectrumLookup`, `OVERLAP_THRESHOLD`, `add_interaction!`, `add_operator!`, `diag!`, `identity_wrap`, `generate_lookup!`, `lookup_exists`, `bare_index`, `dressed_index`, `energy_by_dressed_index`, `energy_by_bare_index`, and `op_in_dressed_eigenbasis`.
                